In [18]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import gsw
import matplotlib.pyplot as plt
from scipy.stats import linregress


# =========================
# 1. Files and parameters
# =========================
npy_dir = Path(r"C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\npy")
fig_dir = Path(r"C:\Users\quzho2904\OneDrive - University of Bergen\GEOF337\report\figures")
fig_dir.mkdir(exist_ok=True)

file_list = [
    "HM2011622.npy",
    "GOS2012115.npy",
    "HM2013624.npy",
    "GOS2014117.npy",
    "HM2015620.npy",
    "HM2016619.npy",
    "GOS2017114.npy",
    "GOS2018111.npy",
    "GOS2019114.npy",
    "GOS2020112.npy",
    "DFN2021460.npy",
    "GOS2022112.npy",
    "GOS2023001013.npy",
    "GOS2024001014.npy",
    "GOS2025001014.npy",
    "HB2026009004.npy",
]

target_station_by_year = {
    2011: "992",
    2012: "444",
    2013: "1084",
    2014: "389",
    2015: "690",
    2016: "951",
    2017: "300",
    2018: "415",
    2019: "385",
    2020: "310",
    2021: "370",
    2022: "367",
    2023: "446",
    2024: "519",
    2025: "736",
    2026: "9",
}

zmin, zmax = 200.0, 400.0

out_csv = npy_dir / "selected_station_200_400m_timeseries_2011_2026.csv"
out_detail_csv = npy_dir / "selected_station_200_400m_detail_2011_2026.csv"

fig1_path = fig_dir / "selected_station_timeseries_O2_sigma0_200_400m_2011_2026.png"
fig2_path = fig_dir / "selected_station_timeseries_CT_S_200_400m_2011_2026.png"


# =========================
# 2. Utility functions
# =========================
def find_key(d, names):
    for n in names:
        if n in d:
            return n
    return None


def arr(x):
    return np.asarray(x, dtype=float)


def extract_year_from_filename(fname):
    m = re.search(r"(20\d{2})", fname)
    if m:
        return int(m.group(1))
    return np.nan


def get_lon_lat(st):
    lon_key = find_key(st, ["LON", "lon", "Longitude", "LONGITUDE"])
    lat_key = find_key(st, ["LAT", "lat", "Latitude", "LATITUDE"])
    if lon_key is None or lat_key is None:
        return np.nan, np.nan

    lon = arr(st[lon_key]).squeeze()
    lat = arr(st[lat_key]).squeeze()
    return float(lon), float(lat)


def get_depth(st):
    k = find_key(st, ["P", "Pressure", "pressure", "PRES", "pres"])
    if k is None:
        return None
    return arr(st[k])


def get_S_T(st):
    S_key = find_key(st, ["S", "SP", "salinity", "Salinity"])
    T_key = find_key(st, ["T", "TEMP", "temperature", "Temperature", "t"])
    if S_key is None or T_key is None:
        raise KeyError("Missing S or T")
    S = arr(st[S_key])
    T = arr(st[T_key])
    return S, T


def get_oxygen(st):
    k = find_key(st, ["raw_o", "OX", "oxygen", "OXYGEN", "DO"])
    if k is None:
        return None, None
    return arr(st[k]), k


def get_sigma0(st):
    k = find_key(st, ["SIGMA0", "sigma0", "Sigma0"])
    if k is None:
        return None
    return arr(st[k])


def fit_linear_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    if np.sum(mask) < 2:
        return np.nan, np.nan, np.nan, np.nan

    res = linregress(x[mask], y[mask])
    slope = res.slope
    intercept = res.intercept
    r2 = res.rvalue ** 2
    pvalue = res.pvalue
    return slope, intercept, r2, pvalue


def find_station_key(ctd_dict, target_station):
    target_station = str(target_station).strip()

    if target_station in ctd_dict:
        return target_station

    for k in ctd_dict.keys():
        if str(k).strip() == target_station:
            return k

    return None


def calculate_endpoint_rate(years, values, start_year, end_year):
    years = np.asarray(years, dtype=float)
    values = np.asarray(values, dtype=float)

    mask = (years >= start_year) & (years <= end_year) & np.isfinite(values)
    yrs_sel = years[mask]
    val_sel = values[mask]

    if len(yrs_sel) < 2:
        return np.nan, np.nan, np.nan, np.nan

    idx = np.argsort(yrs_sel)
    yrs_sel = yrs_sel[idx]
    val_sel = val_sel[idx]

    dyear = yrs_sel[-1] - yrs_sel[0]
    if dyear == 0:
        return np.nan, yrs_sel[0], yrs_sel[-1], np.nan

    rate = (val_sel[-1] - val_sel[0]) / dyear
    return rate, yrs_sel[0], yrs_sel[-1], (val_sel[-1] - val_sel[0])


# =========================
# 3. Single-station calculation
# =========================
def compute_station_means(st):
    lon, lat = get_lon_lat(st)

    z = get_depth(st)
    O_raw, oxygen_key = get_oxygen(st)
    sigma0 = get_sigma0(st)

    if z is None or O_raw is None or sigma0 is None:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "CT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "oxygen_key": oxygen_key,
            "unit_flag": "missing_data",
            "status": "missing_data"
        }

    try:
        S, T = get_S_T(st)
    except Exception:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "CT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "oxygen_key": oxygen_key,
            "unit_flag": "missing_S_or_T",
            "status": "missing_S_or_T"
        }

    n = min(len(z), len(O_raw), len(S), len(T), len(sigma0))

    z = z[:n]
    S = S[:n]
    T = T[:n]
    O_raw = O_raw[:n]
    sigma0 = sigma0[:n]

    valid_basic = (
        np.isfinite(z) &
        np.isfinite(S) &
        np.isfinite(T) &
        np.isfinite(O_raw) &
        np.isfinite(sigma0)
    )

    if np.sum(valid_basic) == 0:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "CT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "oxygen_key": oxygen_key,
            "unit_flag": "all_nan",
            "status": "all_nan"
        }

    z = z[valid_basic]
    S = S[valid_basic]
    T = T[valid_basic]
    O_raw = O_raw[valid_basic]
    sigma0 = sigma0[valid_basic]

    try:
        SA = gsw.SA_from_SP(S, z, lon, lat)
        CT_insitu = gsw.CT_from_t(SA, T, z)
        rho = gsw.rho(SA, CT_insitu, z)
        PT0 = gsw.pt0_from_t(SA, T, z)
    except Exception:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "CT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "oxygen_key": oxygen_key,
            "unit_flag": "gsw_failed",
            "status": "gsw_failed"
        }

    O_mean_raw = np.nanmean(O_raw)

    if O_mean_raw < 20:
        O_mLL = O_raw
        unit_flag = "mL/L"
    else:
        rho_kgL = rho / 1000.0
        O_mLL = O_raw * rho_kgL / 44.66
        unit_flag = "umol/kg_to_mL/L"

    mask = (
        (z >= zmin) &
        (z <= zmax) &
        np.isfinite(O_mLL) &
        np.isfinite(sigma0) &
        np.isfinite(PT0) &
        np.isfinite(S)
    )

    if np.sum(mask) == 0:
        return {
            "lon": lon,
            "lat": lat,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "CT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "oxygen_key": oxygen_key,
            "unit_flag": unit_flag,
            "status": "no_depth_data_200_400m"
        }

    return {
    "lon": lon,
    "lat": lat,
    "O2_200_400m_mLL": np.nanmean(O_mLL[mask]),
    "O2_200_400m_max_mLL": np.nanmax(O_mLL[mask]),
    "sigma0_200_400m": np.nanmean(sigma0[mask]),
    "sigma0_200_400m_max": np.nanmax(sigma0[mask]),
    "CT_200_400m_degC": np.nanmean(PT0[mask]),
    "S_200_400m": np.nanmean(S[mask]),
    "oxygen_key": oxygen_key,
    "unit_flag": unit_flag,
    "status": "ok"
}



# =========================
# 4. Main loop
# =========================
summary = []
detail = []

for fname in file_list:
    fpath = npy_dir / fname
    year = extract_year_from_filename(fname)
    target_station = target_station_by_year.get(year, None)

    if not fpath.exists():
        print(f"\nMissing file: {fname}")
        summary.append({
            "file": fname,
            "year": year,
            "target_station": target_station,
            "station_found": False,
            "lon": np.nan,
            "lat": np.nan,
            "O2_200_400m_mean_mLL": np.nan,
            "sigma0_200_400m_mean": np.nan,
            "CT_200_400m_mean_degC": np.nan,
            "S_200_400m_mean": np.nan,
            "oxygen_key": np.nan,
            "unit_flag": "file_missing",
            "status": "file_missing"
        })
        continue

    if target_station is None:
        print(f"\n{fname}: No target station set for this year")
        summary.append({
            "file": fname,
            "year": year,
            "target_station": np.nan,
            "station_found": False,
            "lon": np.nan,
            "lat": np.nan,
            "O2_200_400m_mean_mLL": np.nan,
            "sigma0_200_400m_mean": np.nan,
            "CT_200_400m_mean_degC": np.nan,
            "S_200_400m_mean": np.nan,
            "oxygen_key": np.nan,
            "unit_flag": "no_target_station_for_year",
            "status": "no_target_station_for_year"
        })
        continue

    ctd = np.load(fpath, allow_pickle=True).item()
    station_key = find_station_key(ctd, target_station)

    if station_key is None:
        print(f"\n{fname}: Target station {target_station} not found")

        detail.append({
            "file": fname,
            "year": year,
            "target_station": target_station,
            "station": np.nan,
            "station_found": False,
            "lon": np.nan,
            "lat": np.nan,
            "O2_200_400m_mLL": np.nan,
            "sigma0_200_400m": np.nan,
            "CT_200_400m_degC": np.nan,
            "S_200_400m": np.nan,
            "oxygen_key": np.nan,
            "unit_flag": "station_not_found",
            "status": "station_not_found"
        })

        summary.append({
            "file": fname,
            "year": year,
            "target_station": target_station,
            "station_found": False,
            "lon": np.nan,
            "lat": np.nan,
            "O2_200_400m_mean_mLL": np.nan,
            "sigma0_200_400m_mean": np.nan,
            "CT_200_400m_mean_degC": np.nan,
            "S_200_400m_mean": np.nan,
            "oxygen_key": np.nan,
            "unit_flag": "station_not_found",
            "status": "station_not_found"
        })
        continue

    st = ctd[station_key]
    result = compute_station_means(st)

    print(
        f"{year}: oxygen variable used = {result['oxygen_key']}, "
        f"unit_flag = {result['unit_flag']}, status = {result['status']}"
    )

    detail.append({
        "file": fname,
        "year": year,
        "target_station": target_station,
        "station": str(station_key),
        "station_found": True,
        "lon": result["lon"],
        "lat": result["lat"],
        "O2_200_400m_mLL": result["O2_200_400m_mLL"],
        "O2_200_400m_max_mLL": result["O2_200_400m_max_mLL"],
        "sigma0_200_400m": result["sigma0_200_400m"],
        "sigma0_200_400m_max": result["sigma0_200_400m_max"],
        "CT_200_400m_degC": result["CT_200_400m_degC"],
        "S_200_400m": result["S_200_400m"],
        "oxygen_key": result["oxygen_key"],
        "unit_flag": result["unit_flag"],
        "status": result["status"]
    })

    summary.append({
        "file": fname,
        "year": year,
        "target_station": target_station,
        "station_found": True,
        "lon": result["lon"],
        "lat": result["lat"],
        "O2_200_400m_mean_mLL": result["O2_200_400m_mLL"],
        "O2_200_400m_max_mLL": result["O2_200_400m_max_mLL"],
        "sigma0_200_400m_mean": result["sigma0_200_400m"],
        "sigma0_200_400m_max": result["sigma0_200_400m_max"],
        "CT_200_400m_mean_degC": result["CT_200_400m_degC"],
        "S_200_400m_mean": result["S_200_400m"],
        "oxygen_key": result["oxygen_key"],
        "unit_flag": result["unit_flag"],
        "status": result["status"]
    })


# =========================
# 5. Save tables
# =========================
df_summary = pd.DataFrame(summary)
df_detail = pd.DataFrame(detail)

df_summary = df_summary.sort_values("year").reset_index(drop=True)
df_detail = df_detail.sort_values(["year"]).reset_index(drop=True)

df_summary.to_csv(out_csv, index=False, encoding="utf-8-sig")
df_detail.to_csv(out_detail_csv, index=False, encoding="utf-8-sig")

print("\nSaved successfully:")
print(out_csv)
print(out_detail_csv)


# =========================
# 6. Trend fitting
# =========================
years = df_summary["year"].values.astype(float)

O2 = df_summary["O2_200_400m_mean_mLL"].values.astype(float)
sigma0 = df_summary["sigma0_200_400m_mean"].values.astype(float)
CT = df_summary["CT_200_400m_mean_degC"].values.astype(float)
S = df_summary["S_200_400m_mean"].values.astype(float)
O2_max = df_summary["O2_200_400m_max_mLL"].values.astype(float)
sigma0_max = df_summary["sigma0_200_400m_max"].values.astype(float)

mask_fit_sigma = (years >= 2011) & (years <= 2026) & np.isfinite(sigma0)
sigma_slope, sigma_intercept, sigma_r2, sigma_p = fit_linear_stats(
    years[mask_fit_sigma],
    sigma0[mask_fit_sigma]
)

mask_fit_CT = (years >= 2011) & (years <= 2026) & np.isfinite(CT)
CT_slope, CT_intercept, CT_r2, CT_p = fit_linear_stats(
    years[mask_fit_CT],
    CT[mask_fit_CT]
)

mask_fit_S = (years >= 2011) & (years <= 2026) & np.isfinite(S)
S_slope, S_intercept, S_r2, S_p = fit_linear_stats(
    years[mask_fit_S],
    S[mask_fit_S]
)

print("\nRegression results for 2011–2026:")
print(f"sigma0 slope = {sigma_slope:.6f} /yr, R2 = {sigma_r2:.4f}, p = {sigma_p:.4f}")
print(f"CT slope     = {CT_slope:.6f} degC/yr, R2 = {CT_r2:.4f}, p = {CT_p:.4f}")
print(f"S slope      = {S_slope:.6f} /yr, R2 = {S_r2:.4f}, p = {S_p:.4f}")


# =========================
# 7. Direct calculation of annual change
# =========================
periods = [
    (2013, 2017),
    (2021, 2024),
]

O2_rate_list = []
sigma_rate_list = []

print("\n==============================")
print("Direct annual change (endpoint method)")
print("==============================")

for start_year, end_year in periods:
    dO2_dt, y1_o2, y2_o2, dO2 = calculate_endpoint_rate(years, O2, start_year, end_year)
    dsigma_dt, y1_sg, y2_sg, dsigma = calculate_endpoint_rate(years, sigma0, start_year, end_year)

    print(f"\nPeriod: {start_year}–{end_year}")

    if np.isfinite(dO2_dt):
        print(f"Change in oxygen (mL O2 L^-1 yr^-1): {dO2_dt:.6f}")
        print(f"Total oxygen change over period:      {dO2:.6f}")
        O2_rate_list.append(dO2_dt)
    else:
        print("Change in oxygen: insufficient data")

    if np.isfinite(dsigma_dt):
        print(f"Annual change in sigma0:              {dsigma_dt:.6f}")
        print(f"Total sigma0 change over period:      {dsigma:.6f}")
        sigma_rate_list.append(dsigma_dt)
    else:
        print("Annual change in sigma0: insufficient data")

if len(O2_rate_list) > 0:
    mean_dO2_dt = np.nanmean(O2_rate_list)
    mean_abs_dO2_dt = np.nanmean(np.abs(O2_rate_list))

    print("\n------------------------------")
    print("Mean oxygen annual change from the two periods:")
    print(f"Signed mean = {mean_dO2_dt:.6f} mL O2 L^-1 yr^-1")
    print(f"Absolute mean = {mean_abs_dO2_dt:.6f} mL O2 L^-1 yr^-1")
else:
    mean_dO2_dt = np.nan
    mean_abs_dO2_dt = np.nan
    print("\nMean oxygen annual change: insufficient data")

if len(sigma_rate_list) > 0:
    mean_dsigma_dt = np.nanmean(sigma_rate_list)
    mean_abs_dsigma_dt = np.nanmean(np.abs(sigma_rate_list))

    print("\nMean sigma0 annual change from the two periods:")
    print(f"Signed mean = {mean_dsigma_dt:.6f} yr^-1")
    print(f"Absolute mean = {mean_abs_dsigma_dt:.6f} yr^-1")
else:
    mean_dsigma_dt = np.nan
    mean_abs_dsigma_dt = np.nan
    print("\nMean sigma0 annual change: insufficient data")

print("==============================")


# =========================
# 8. Plot 1: O2 + sigma0
# =========================
fig, ax1 = plt.subplots(figsize=(9, 6), dpi=200)

mask1 = np.isfinite(years) & np.isfinite(O2)
ax1.plot(
    years[mask1], O2[mask1],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="tab:blue",
    label="DO mean"
)

ax1.set_xlabel("Year")
ax1.set_ylabel(r"DO (mL L$^{-1}$)", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()

mask1max = np.isfinite(years) & np.isfinite(O2_max)
ax1.plot(
    years[mask1max], O2_max[mask1max],
    marker="^",
    linestyle="--",
    markersize=6,
    color="tab:blue",
    label="DO max"
)

mask2 = np.isfinite(years) & np.isfinite(sigma0)
ax2.plot(
    years[mask2], sigma0[mask2],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="tab:red",
    label=r"$\sigma_0$ mean"
)

mask2max = np.isfinite(years) & np.isfinite(sigma0_max)
ax2.plot(
    years[mask2max], sigma0_max[mask2max],
    marker="^",
    linestyle="--",
    markersize=6,
    color="tab:red",
    label=r"$\sigma_0$ max"

)

if np.sum(mask_fit_sigma) >= 2:
    xfit_sigma = years[mask_fit_sigma]
    sort_idx = np.argsort(xfit_sigma)
    xfit_sigma = xfit_sigma[sort_idx]
    yfit_sigma = sigma_slope * xfit_sigma + sigma_intercept

    ax2.plot(
        xfit_sigma,
        yfit_sigma,
        linestyle=":",
        linewidth=1.8,
        color="tab:red",
        label=r"$\sigma_0$ trend"
    )

ax2.set_ylabel(r"$\sigma_0$ (kg m$^{-3}$)", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
ax1.set_ylim(2.0, 5.5)
ax2.set_ylim(27.16, 27.38)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2, loc="best", fontsize=9)
ax1.text(0.02, 0.95, "(a)", transform=ax1.transAxes, fontsize=13, fontweight="bold", va="top")
plt.title(r"Masfjorden selected-station DO and $\sigma_0$ (200–400 m)")
plt.tight_layout()
plt.savefig(fig1_path, dpi=300, bbox_inches="tight")
plt.close()


# =========================
# 9. Plot 2: CT + S
# =========================
fig, ax1 = plt.subplots(figsize=(9, 6), dpi=200)

mask3 = np.isfinite(years) & np.isfinite(CT)
ax1.plot(
    years[mask3], CT[mask3],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="tab:green",
    label=r"$\Theta$ mean"
)

if np.sum(mask_fit_CT) >= 2:
    xfit_CT = years[mask_fit_CT]
    sort_idx = np.argsort(xfit_CT)
    xfit_CT = xfit_CT[sort_idx]
    yfit_CT = CT_slope * xfit_CT + CT_intercept

    ax1.plot(
        xfit_CT,
        yfit_CT,
        linestyle=":",
        linewidth=1.5,
        color="tab:green",
        label=r"$\Theta$ trend"
    )

ax1.set_xlabel("Year")
ax1.set_ylabel(r"$\Theta$ (°C)", color="tab:green")
ax1.tick_params(axis="y", labelcolor="tab:green")
ax1.set_ylim(8.05, 8.40)

ax2 = ax1.twinx()

mask4 = np.isfinite(years) & np.isfinite(S)
ax2.plot(
    years[mask4], S[mask4],
    marker="o",
    linestyle="-",
    linewidth=1.8,
    color="tab:purple",
    label=r"$S_A$ mean"
)

if np.sum(mask_fit_S) >= 2:
    xfit_S = years[mask_fit_S]
    sort_idx = np.argsort(xfit_S)
    xfit_S = xfit_S[sort_idx]
    yfit_S = S_slope * xfit_S + S_intercept

    ax2.plot(
        xfit_S,
        yfit_S,
        linestyle=":",
        linewidth=1.5,
        color="tab:purple",
        label=r"$S_A$ trend"
    )

ax2.set_ylabel(r"$S_A$ (g kg$^{-1}$)", color="tab:purple")
ax2.tick_params(axis="y", labelcolor="tab:purple")
ax2.set_ylim(34.90, 35.20)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2, loc="best", fontsize=9)
ax1.text(0.02, 0.95, "(b)", transform=ax1.transAxes, fontsize=13, fontweight="bold", va="top")
plt.title(r"Masfjorden selected-station $\Theta$ and $S_A$ (200–400 m)")
plt.tight_layout()
plt.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.close()

print("\nFigures saved:")
print(fig1_path)
print(fig2_path)


# =========================
# 10. Calculate annual Vn
# =========================
Os = 5.4
delta_t = 1.0

O2_series = df_summary["O2_200_400m_mean_mLL"].values.astype(float)
years = df_summary["year"].values.astype(float)

Vn_list = [np.nan]

for i in range(1, len(O2_series)):
    OB_n = O2_series[i]
    OB_prev = O2_series[i - 1]

    if not (np.isfinite(OB_n) and np.isfinite(OB_prev)):
        Vn_list.append(np.nan)
        continue

    numerator = OB_n - OB_prev + mean_abs_dO2_dt * delta_t
    denominator = (Os - OB_prev) * delta_t

    if denominator == 0:
        Vn = np.nan
    else:
        Vn = numerator / denominator

    Vn_list.append(Vn)

df_summary["Vn"] = Vn_list

print("\nVn calculation complete")
print("b used =", mean_abs_dO2_dt)
print("Os =", Os)

print("\nVn preview:")
print(df_summary[["year", "O2_200_400m_mean_mLL", "Vn"]])


# =========================
# 11. Save Vn time series
# =========================
out_vn_csv = npy_dir / "Vn_timeseries_2011_2026.csv"

df_vn = df_summary[
    ["year", "O2_200_400m_mean_mLL", "Vn"]
].copy()

df_vn.columns = [
    "year",
    "O2_200_400m_mLL",
    "Vn"
]

df_vn.to_csv(
    out_vn_csv,
    index=False,
    encoding="utf-8-sig"
)

print("\nVn saved:")
print(out_vn_csv)


# =========================
# 12. Total changes only
# =========================
periods_total_change = [
    (2017, 2019),
    (2020, 2021),
    (2024, 2025),
    (2013, 2017),
    (2021, 2024),
]

print("\n==============================")
print("Total changes in O2 and sigma0")
print("==============================")

for start_year, end_year in periods_total_change:
    _, _, _, dO2 = calculate_endpoint_rate(years, O2, start_year, end_year)
    _, _, _, dsigma = calculate_endpoint_rate(years, sigma0, start_year, end_year)

    print(f"\nPeriod {start_year}-{end_year}")

    if np.isfinite(dO2):
        print(f"Total O2 change (mL/L): {dO2:.6f}")
    else:
        print("Insufficient O2 data")

    if np.isfinite(dsigma):
        print(f"Total sigma0 change: {dsigma:.6f}")
    else:
        print("Insufficient sigma0 data")
    

# =========================
# 13. Changes in DO max and sigma0 max
# =========================
periods_DOmax_change = [
    (2013, 2015),
    (2015, 2016),
    (2017, 2019),
    (2020, 2021),
    (2021, 2023),
    (2023, 2024),
]

print("\n==============================")
print("Total changes in DO max")
print("==============================")

for start_year, end_year in periods_DOmax_change:
    _, _, _, dO2max = calculate_endpoint_rate(years, O2_max, start_year, end_year)

    print(f"\nPeriod {start_year}-{end_year}")

    if np.isfinite(dO2max):
        print(f"Total DO max change (mL/L): {dO2max:.6f}")
    else:
        print("Insufficient DO max data")


print("\n==============================")
print("Total change in sigma0 max")
print("==============================")

_, _, _, dsigma0max_2425 = calculate_endpoint_rate(years, sigma0_max, 2024, 2025)

if np.isfinite(dsigma0max_2425):
    print(f"Period 2024-2025")
    print(f"Total sigma0 max change (kg m^-3): {dsigma0max_2425:.6f}")
else:
    print("Period 2024-2025")
    print("Insufficient sigma0 max data")

2011: oxygen variable used = OX, unit_flag = mL/L, status = ok
2012: oxygen variable used = OX, unit_flag = mL/L, status = ok
2013: oxygen variable used = OX, unit_flag = mL/L, status = ok
2014: oxygen variable used = OX, unit_flag = mL/L, status = ok
2015: oxygen variable used = OX, unit_flag = mL/L, status = ok
2016: oxygen variable used = OX, unit_flag = mL/L, status = ok
2017: oxygen variable used = OX, unit_flag = mL/L, status = ok
2018: oxygen variable used = OX, unit_flag = mL/L, status = ok
2019: oxygen variable used = OX, unit_flag = umol/kg_to_mL/L, status = ok
2020: oxygen variable used = OX, unit_flag = umol/kg_to_mL/L, status = ok
2021: oxygen variable used = OX, unit_flag = umol/kg_to_mL/L, status = ok
2022: oxygen variable used = OX, unit_flag = umol/kg_to_mL/L, status = ok
2023: oxygen variable used = OX, unit_flag = umol/kg_to_mL/L, status = ok
2024: oxygen variable used = OX, unit_flag = umol/kg_to_mL/L, status = ok
2025: oxygen variable used = OX, unit_flag = umol/kg